In [3]:
import pandas as pd
import json

# Load historical Amazon support conversations
df = pd.read_csv("amazon_pairs.csv")

# Load allowed intents
with open("amazon_intents.json", "r", encoding="utf-8") as f:
    INTENTS = json.load(f)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nIntents:")
for intent, description in INTENTS.items():
    print(f"- {intent}: {description}")

Dataset shape: (168823, 2)

Columns: ['customer_message', 'amazon_response']

Intents:
- Delivery & Tracking: Missing, late, mis-delivered packages, order status, or tracking questions.
- Returns & Refunds: Returns, refunds, refund delays, or problems with the return/refund process.
- Account Access & Security: Login, password, account access, account lockout, or suspicious account activity.
- Prime Membership & Benefits: Questions or problems related to Prime membership, subscription, eligibility, or Prime benefits.
- Promotions & Offers: Questions or problems involving discounts, promotional offers, contests, giveaways, or promotional eligibility.
- Digital Content & Device Issues: Problems with Kindle, Audible, Prime Video, digital content, Echo, Fire TV, or other Amazon devices/services.
- Product Quality & Listing: Damaged, defective, incomplete, incorrect, or mis-described products and inaccurate listings.
- Payment & Billing: Incorrect charges, payment problems, cashback, Amazon

In [4]:
print(df.isnull().sum())

customer_message    0
amazon_response     0
dtype: int64


There are no null values which is good

In [5]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content=row["customer_message"],
        metadata={
            "amazon_response": row["amazon_response"]
        }
    )
    for _, row in df.iterrows()
]

In [6]:
print(documents[0])

page_content='amazonのfireTVstickが見れない😢' metadata={'amazon_response': '@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET'}


This is nice because we are searching for the customer messages

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [ ]:
#creating a 1000 row test dataset
from langchain_core.documents import Document

test_df = df.sample(1000, random_state=42).reset_index(drop=True)


In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
texts = test_df["customer_message"].tolist()

test_embeddings = embeddings.embed_documents(texts)

print(len(test_embeddings))
print(len(test_embeddings[0]))

1000
384


In [13]:
small_df = df.sample(100, random_state=42).reset_index(drop=True)

small_texts = small_df["customer_message"].tolist()

small_embeddings = embeddings.embed_documents(small_texts)

print(len(small_embeddings))
print(len(small_embeddings[0]))

100
384


In [14]:
from langchain_community.vectorstores import FAISS

small_documents = [
    Document(
        page_content=row["customer_message"],
        metadata={
            "amazon_response": row["amazon_response"]
        }
    )
    for _, row in small_df.iterrows()
]


In [15]:

vector_store = FAISS.from_documents(
    small_documents,
    embeddings
)

print("FAISS created!")

FAISS created!


In [17]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

In [18]:
#testing 
query = "My package hasn't arrived yet. Can you tell me where it is?"

results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"RESULT {i}")
    print(f"{'='*60}")
    print("Customer:", doc.page_content)
    print("Amazon:", doc.metadata["amazon_response"])


RESULT 1
Customer: Just found an @115821 parcel in my front garden... where’s the strangest “safe place” you’ve ever found a parcel? 😂 https://t.co/YGGCeC8Oyc
Amazon: @350701 Hi Carl, we'd like to investigate it. Please submit this secure form ans we'll be in touch: https://t.co/tkLCr7DNil ^AT

RESULT 2
Customer: @AmazonHelp My ordered is not delivered yet?
Amazon: @444309 I'm sorry for the miss, we have sent you a correspondence here: https://t.co/DTSNmGldJf kindly check and revert. ^AH

RESULT 3
Customer: @AmazonHelp - Order No 407-6706063-7981118. God know when will you guys delivery. Terrible Service, no one knows the status.
Amazon: @383173 Please don’t provide your order details as it is personal information. Our Twitter page is visible to the public. ^MS

RESULT 4
Customer: @AmazonHelp Order no. 406-6787527-3269912 dated 12-10-17 not yet dispatch??? Whn can I get it? Its more than 10 days now..
Amazon: @364452 Please don't provide your order details, as we consider it to be per

In [19]:
print("Total rows:", len(df))
print("Customer messages:", df["customer_message"].notna().sum())
print("Amazon responses:", df["amazon_response"].notna().sum())

Total rows: 168823
Customer messages: 168823
Amazon responses: 168823


In [21]:
#estimating the time for embedding 
import time

sample_texts = df["customer_message"].head(1000).tolist()

start = time.time()

_ = embeddings.embed_documents(sample_texts)

elapsed = time.time() - start

print(f"1000 embeddings took {elapsed:.2f} seconds")
print(f"Estimated time for 168823: {(elapsed * len(df) / 1000) / 60:.2f} minutes")

1000 embeddings took 9.54 seconds
Estimated time for 168823: 26.86 minutes


In [22]:
#creating embedding by using a safe approach in case the kernel gets intrupted
import numpy as np
import pickle
from pathlib import Path

BATCH_SIZE = 1000
CHECKPOINT_EVERY = 10

checkpoint_dir = Path("amazon_embedding_checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

texts = df["customer_message"].tolist()

total = len(texts)

# Find latest checkpoint
checkpoint_files = sorted(
    checkpoint_dir.glob("checkpoint_*.pkl")
)

if checkpoint_files:

    latest_checkpoint = checkpoint_files[-1]

    print("Loading:", latest_checkpoint)

    with open(latest_checkpoint, "rb") as f:
        checkpoint = pickle.load(f)

    all_embeddings = checkpoint["embeddings"]
    start_index = checkpoint["next_index"]

    print(
        f"Resuming from message {start_index:,}"
    )

else:

    all_embeddings = []
    start_index = 0

    print("Starting from beginning.")


while start_index < total:

    end_index = min(
        start_index + BATCH_SIZE,
        total
    )

    batch_texts = texts[start_index:end_index]

    print(
        f"Embedding {start_index:,} → "
        f"{end_index:,} / {total:,}"
    )

    batch_embeddings = embeddings.embed_documents(
        batch_texts
    )

    batch_embeddings = np.array(
        batch_embeddings,
        dtype=np.float32
    )

    all_embeddings.append(batch_embeddings)

    start_index = end_index

    # Save every 10,000 messages
    if (
        start_index % (BATCH_SIZE * CHECKPOINT_EVERY) == 0
        or start_index == total
    ):

        checkpoint = {
            "embeddings": all_embeddings,
            "next_index": start_index
        }

        checkpoint_path = (
            checkpoint_dir /
            f"checkpoint_{start_index}.pkl"
        )

        with open(checkpoint_path, "wb") as f:
            pickle.dump(checkpoint, f)

        print(
            f"Checkpoint saved at "
            f"{start_index:,} messages"
        )

print("All embeddings generated!")

Starting from beginning.
Embedding 0 → 1,000 / 168,823
Embedding 1,000 → 2,000 / 168,823
Embedding 2,000 → 3,000 / 168,823
Embedding 3,000 → 4,000 / 168,823
Embedding 4,000 → 5,000 / 168,823
Embedding 5,000 → 6,000 / 168,823
Embedding 6,000 → 7,000 / 168,823
Embedding 7,000 → 8,000 / 168,823
Embedding 8,000 → 9,000 / 168,823
Embedding 9,000 → 10,000 / 168,823
Checkpoint saved at 10,000 messages
Embedding 10,000 → 11,000 / 168,823
Embedding 11,000 → 12,000 / 168,823
Embedding 12,000 → 13,000 / 168,823
Embedding 13,000 → 14,000 / 168,823
Embedding 14,000 → 15,000 / 168,823
Embedding 15,000 → 16,000 / 168,823
Embedding 16,000 → 17,000 / 168,823
Embedding 17,000 → 18,000 / 168,823
Embedding 18,000 → 19,000 / 168,823
Embedding 19,000 → 20,000 / 168,823
Checkpoint saved at 20,000 messages
Embedding 20,000 → 21,000 / 168,823
Embedding 21,000 → 22,000 / 168,823
Embedding 22,000 → 23,000 / 168,823
Embedding 23,000 → 24,000 / 168,823
Embedding 24,000 → 25,000 / 168,823
Embedding 25,000 → 26,000 

In [26]:
import faiss
embedding_matrix = np.vstack(all_embeddings).astype("float32")

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("Embedding shape:", embedding_matrix.shape)
print("FAISS vectors:", index.ntotal)

Embedding shape: (168823, 384)
FAISS vectors: 168823


In [27]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=None,
    index_to_docstore_id={}
)

In [28]:
print(type(all_embeddings))
print(len(all_embeddings))

embedding_matrix = np.vstack(all_embeddings).astype("float32")

print("Shape:", embedding_matrix.shape)

<class 'list'>
169
Shape: (168823, 384)


In [29]:
np.save("amazon_embeddings.npy", embedding_matrix)